# LIBERO eval — **최종 결과 정리** (500ep 완료만)

최종 eval 중 **500ep 도달한 run 만** 골라 확정 표를 만든다. (중간 스냅샷은 `eval_progress`)

- **500ep 미만/미완 run 은 자동 제외** + 제외 목록 출력 → 표에 안 섞임.
- SR 표(model×seed + mean±std + pooled Wilson 95% CI), 떨림 표(jerk/LDJ/SPARC/SignFlip),
  최종 표 PNG, 요약 MD → **zip 하나** = `outputs/final/share/libero_10_final_<시각>.zip`.
- 읽기 전용. seed3 까지 다 끝난 뒤 돌리면 그게 최종본.


In [ ]:
import sys, json, csv, time
from pathlib import Path
from collections import defaultdict
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
import numpy as np

# ── 최종 LIBERO eval 대상 ──
MODELS = ['acm', 'act', 'bimamba', 'bimamba_s7']   # 있으면 추가로 발견된 모델도 표에 포함
SEEDS  = [0, 1, 2, 3]
TASK   = 'libero_10'
TARGET_EP = cf.EVAL_N_EP                 # 500 — 이 값 이상 도달한 run '만' 최종 집계

ROOT = cf.OUTPUT_BASE / 'eval_clean' / TASK
OUT  = cf.OUTPUT_BASE / 'share' / f'{TASK}_final'
OUT.mkdir(parents=True, exist_ok=True)
STAMP = time.strftime('%Y%m%d_%H%M')
print('scan:', ROOT, '| exists:', ROOT.is_dir())
print(f'최종 집계 기준: {TARGET_EP}ep 도달 run 만  |  스냅샷: {STAMP}')

## 1) 스캔 — 500ep 완료 run 만 채택


In [ ]:
# ── eval_info.json 스캔 → 500ep 도달한 것만 채택, 미만/미완은 제외 목록으로 ──
def parse_info(p):
    try:
        d = json.loads(p.read_text())
    except Exception:
        return None
    ov = d.get('overall', d)
    sr = ov.get('pc_success')
    if sr is None and 'success' in ov:
        sr = 100.0 * float(np.mean(ov['success']))
    return {'sr': sr, 'n_ep': ov.get('n_ep', ov.get('n_episodes'))}

done, excluded = {}, []          # done[(model,seed)] = {...}  ;  excluded = [(model,seed,이유)]
if ROOT.is_dir():
    for info in sorted(ROOT.rglob('eval_info.json')):
        parts = info.parent.relative_to(ROOT).parts
        model = parts[0] if parts else '?'
        seed = next((int(p[4:]) for p in parts if p.startswith('seed') and p[4:].isdigit()), None)
        m = parse_info(info)
        if m is None or seed is None or m['sr'] is None:
            continue
        n_ep = m['n_ep'] or 0
        rec = {'sr': m['sr'], 'n_ep': n_ep, 'path': str(info.parent.relative_to(ROOT)),
               'has_actions': (info.parent / 'actions').is_dir()}
        if n_ep >= TARGET_EP:
            done[(model, seed)] = rec           # 500ep 도달 → 채택 (같은 셀 여러 rep 이면 마지막 것)
        else:
            excluded.append((model, seed, f'{n_ep}ep < {TARGET_EP}'))

# 표에 넣을 모델 = 지정 + 발견된 것(순서 유지)
found = [m for m, _ in done]
MODELS_ALL = MODELS + [m for m in dict.fromkeys(found) if m not in MODELS]
for m in MODELS_ALL:
    cf.v23.MODEL_DIR_NAMES.setdefault(m, m)

print(f'{TARGET_EP}ep 완료 run: {len(done)}개')
if excluded:
    print(f'제외(500ep 미만) {len(excluded)}개:', excluded)
else:
    print('제외 없음 — 발견된 run 전부 500ep ✅')
print('표 모델:', MODELS_ALL)

## 2) SR 표 (model × seed, mean±std, pooled 95% CI)


In [ ]:
# ── SR 표: model × seed (500ep 완료만) + mean±std + pooled Wilson 95% CI ──
hdr = f"{'model':<14}" + ''.join(f'{("seed"+str(s)):>8}' for s in SEEDS) + f"{'mean':>8}{'±std':>7}{'pooled 95%CI':>16}"
print(hdr); print('-' * len(hdr))
sr_rows = []
for m in MODELS_ALL:
    per = {s: done[(m, s)]['sr'] for s in SEEDS if (m, s) in done}
    vals = list(per.values())
    mean = float(np.mean(vals)) if vals else None
    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else (0.0 if vals else None)
    # pooled: 성공수/에피소드수 합산
    pk = pn = 0
    for s in SEEDS:
        if (m, s) in done:
            n = int(done[(m, s)]['n_ep']); pk += int(round(done[(m, s)]['sr'] / 100.0 * n)); pn += n
    ci = ''
    if pn:
        lo, hi = cf.v23.wilson_ci(pk, pn)
        ci = f'[{lo*100:.1f}, {hi*100:.1f}]'
    cells = ''.join((f'{per[s]:>8.1f}' if s in per else f'{"·":>8}') for s in SEEDS)
    print(f'{m:<14}{cells}' + (f'{mean:>8.1f}' if mean is not None else f'{"-":>8}')
          + (f'{std:>7.1f}' if std is not None else f'{"-":>7}') + f'{ci:>16}')
    row = {'model': m, **{f'seed{s}': (round(per[s], 1) if s in per else None) for s in SEEDS},
           'mean': (round(mean, 2) if mean is not None else None),
           'std': (round(std, 2) if std is not None else None),
           'n_seed': len(vals), 'pooled_sr': (round(pk / pn * 100, 2) if pn else None),
           'pooled_ci': ci}
    sr_rows.append(row)

cols = ['model'] + [f'seed{s}' for s in SEEDS] + ['mean', 'std', 'n_seed', 'pooled_sr', 'pooled_ci']
with open(OUT / f'sr_table_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(sr_rows)
print('\nsaved:', OUT / f'sr_table_{STAMP}.csv')

## 3) 떨림 표 — jerk RMS · LDJ · SPARC · Sign Flip


In [ ]:
# ── 떨림 표: jerk RMS · LDJ · SPARC · Sign Flip (500ep 완료 + action.pt 있는 것 pool) ──
trajs = {}
for (m, s), d in done.items():
    if not d['has_actions']:
        continue
    tr = cf.v23._load_action_trajs(ROOT / d['path'] / 'actions') or []
    if tr:
        trajs.setdefault(m, []).extend(tr)

smooth = {}
if not trajs:
    print('action(.pt) 없음 — SR 표만')
else:
    import smooth_metrics as sm
    print(f"{'model':<14}{'jerk_RMS':>10}{'LDJ':>10}{'SPARC':>10}{'sign_flip':>11}{'n_traj':>8}")
    print('  (smoother =    lower      0에근접    0에근접       lower)')
    print('-' * 63)
    srows = []
    for m in MODELS_ALL:
        if m not in trajs:
            continue
        agg = sm.aggregate_smoothness(trajs[m], chunk_size=100, fs=cf.fps_of(TASK))
        smooth[m] = agg
        print(f"{m:<14}{agg['jerk_rms_mean']:>10.5f}{agg['ldj_mean']:>10.3f}"
              f"{agg['sparc_mean']:>10.3f}{agg['sign_flips_mean']:>11.4f}{len(trajs[m]):>8}")
        srows.append({'model': m,
                      'jerk_rms': round(agg['jerk_rms_mean'], 5), 'jerk_rms_std': round(agg['jerk_rms_std'], 5),
                      'LDJ': round(agg['ldj_mean'], 3), 'SPARC': round(agg['sparc_mean'], 3),
                      'sign_flip': round(agg['sign_flips_mean'], 4), 'n_traj': len(trajs[m])})
    with open(OUT / f'smoothness_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['model', 'jerk_rms', 'jerk_rms_std', 'LDJ', 'SPARC', 'sign_flip', 'n_traj'])
        w.writeheader(); w.writerows(srows)
    print('\nsaved:', OUT / f'smoothness_{STAMP}.csv')

## 4) 최종 표 이미지 (SR + 떨림 한 장)


In [ ]:
# ── 최종 표 PNG: 모델 × [SR mean±std · n_seed · jerk · LDJ · SPARC · SignFlip] ──
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
col_lbl = ['SR mean±std', 'n_seed', 'jerk_RMS', 'LDJ', 'SPARC', 'sign_flip']
rows_txt, rlabels = [], []
for r in sr_rows:
    m = r['model']
    sr = '-' if r['mean'] is None else f"{r['mean']:.1f} ± {r['std']:.1f}"
    ag = smooth.get(m)
    if ag:
        cells = [sr, f"{r['n_seed']}/{len(SEEDS)}", f"{ag['jerk_rms_mean']:.4f}",
                 f"{ag['ldj_mean']:.2f}", f"{ag['sparc_mean']:.2f}", f"{ag['sign_flips_mean']:.3f}"]
    else:
        cells = [sr, f"{r['n_seed']}/{len(SEEDS)}", '-', '-', '-', '-']
    rows_txt.append(cells); rlabels.append(m)

fig, ax = plt.subplots(figsize=(1.8 + 1.5 * len(col_lbl), 0.7 + 0.5 * len(rlabels)))
ax.axis('off')
t = ax.table(cellText=rows_txt, rowLabels=rlabels, colLabels=col_lbl, loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1, 1.6)
ax.set_title(f'LIBERO-10 final ({TARGET_EP}ep)   {STAMP}', fontsize=12, pad=12)
png = OUT / f'final_table_{STAMP}.png'
fig.savefig(png, dpi=150, bbox_inches='tight'); plt.close(fig)
print('saved:', png)

## 5) zip → 팀에 이 파일 하나


In [ ]:
# ── 요약 MD + zip → 팀에 이 파일 하나 ──
lines = [f'# LIBERO-10 최종 결과 ({TARGET_EP}ep, {STAMP})', '',
         f'- 채택 run: {TARGET_EP}ep 도달한 것만 ({len(done)}개)', '']
if excluded:
    lines += [f'- ⚠️ 제외(500ep 미만): {excluded}', '']
lines += ['## SR (mean±std over seeds, pooled 95% CI)', '']
for r in sr_rows:
    if r['mean'] is not None:
        lines.append(f"- {r['model']}: {r['mean']:.1f} ± {r['std']:.1f}  "
                     f"(n_seed {r['n_seed']}/{len(SEEDS)}, pooled {r['pooled_sr']} {r['pooled_ci']})")
if smooth:
    lines += ['', '## 떨림 (jerk RMS ↓ / LDJ,SPARC 0근접 / SignFlip ↓)', '']
    for m, ag in smooth.items():
        lines.append(f"- {m}: jerk {ag['jerk_rms_mean']:.4f} · LDJ {ag['ldj_mean']:.2f} · "
                     f"SPARC {ag['sparc_mean']:.2f} · signflip {ag['sign_flips_mean']:.3f}")
(OUT / f'README_{STAMP}.md').write_text('\n'.join(lines), encoding='utf-8')

import shutil
zip_path = shutil.make_archive(str(cf.OUTPUT_BASE / 'share' / f'{TASK}_final_{STAMP}'), 'zip', root_dir=OUT)
print('보낼 파일:', zip_path, '\n')
for p in sorted(OUT.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(OUT)}  ({p.stat().st_size/1024:.0f} KB)')